# Расчет коэффициентов пролонгации за 2023 год

Этот ноутбук реализует требования ТЗ по пролонгациям:

- подготовка и очистка данных,
- расчет коэффициентов пролонгации 1-го и 2-го месяца,
- расчет по менеджерам и по отделу,
- расчет годовых взвешенных коэффициентов,
- экспорт результата в Excel-отчет.


## Шаг 1. Импорты и загрузка данных

Используем `prolongations.csv` как источник месяца завершения проекта и ответственного менеджера (`AM`), а `financial_data.csv` — как источник отгрузок.


In [1]:
from pathlib import Path
import pandas as pd

from build_prolongation_report import (
    parse_ru_month,
    build_project_month_agg,
    calculate_mart,
    monthly_kpi_for_group,
    yearly_weighted_kpi,
)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
OUT_DIR = BASE_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

prolongations = pd.read_csv(DATA_DIR / 'prolongations.csv')
financial = pd.read_csv(DATA_DIR / 'financial_data.csv')

print('prolongations shape:', prolongations.shape)
print('financial shape:', financial.shape)
prolongations.head()

prolongations shape: (477, 3)
financial shape: (451, 19)


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


## Шаг 2. Очистка и построение проектной витрины

Ключевые правила ТЗ в логике:

- `стоп` / `end`: если встречается в последнем месяце реализации проекта или раньше, проект исключается из пролонгаций;
- `в ноль`: трактуется как 0, но для `last_month_shipment` берем предыдущий месяц **только если** в месяце завершения все части оплаты равны 0;
- дубликаты `id` суммируются по проекту и месяцу.


In [2]:
month_cols = [c for c in financial.columns if c not in {'id', 'Причина дубля', 'Account'}]
month_cols_sorted = sorted(month_cols, key=parse_ru_month)

month_agg = build_project_month_agg(financial, month_cols_sorted)
mart = calculate_mart(prolongations, month_agg, month_cols_sorted)

print('mart shape:', mart.shape)
mart.head()

mart shape: (477, 9)


,id,AM,completion_month_ts,completion_month,last_month_shipment,ship_t1,ship_t2,excluded_stop_end,used_prev_month_for_last
0,42,Васильев Артем Александрович,2022-11-01,Ноябрь 2022,36220.0,0.0,0.0,False,False
1,453,Васильев Артем Александрович,2022-11-01,Ноябрь 2022,0.0,39245.0,44320.0,False,False
2,548,Михайлов Андрей Сергеевич,2022-11-01,Ноябрь 2022,674000.0,674000.0,674000.0,False,False
3,87,Соколова Анастасия Викторовна,2022-11-01,Ноябрь 2022,70050.0,0.0,73380.0,False,False
4,429,Соколова Анастасия Викторовна,2022-11-01,Ноябрь 2022,30280.0,35580.0,35830.0,False,False


## Шаг 3. Расчет KPI по менеджерам и отделу

Для каждого отчетного месяца 2023 считаем:

- `coef_1m = numerator_1m / denominator_1m`, где
  - `denominator_1m`: сумма последнего месяца проектов, завершившихся в прошлом месяце,
  - `numerator_1m`: сумма отгрузок текущего месяца по тем же проектам, где есть пролонгация в 1-й месяц.

- `coef_2m = numerator_2m / denominator_2m`, где
  - база: проекты, завершившиеся 2 месяца назад и не пролонгированные в 1-й месяц,
  - `denominator_2m`: сумма последнего месяца этой базы,
  - `numerator_2m`: сумма отгрузок текущего месяца по проектам базы, пролонгированным во 2-й месяц.

Годовые коэффициенты считаются как взвешенные (отношение сумм), а не среднее месячных коэффициентов.


In [3]:
report_months = [pd.Timestamp(year=2023, month=m, day=1) for m in range(1, 13)]

manager_monthly_parts = []
manager_yearly_parts = []
for manager, g in mart.groupby('AM'):
    monthly = monthly_kpi_for_group(g, report_months)
    monthly.insert(0, 'AM', manager)
    manager_monthly_parts.append(monthly)
    manager_yearly_parts.append(yearly_weighted_kpi(monthly, group_name=manager, year=2023))

manager_monthly = pd.concat(manager_monthly_parts, ignore_index=True).sort_values(['AM', 'report_month'])
manager_yearly = pd.concat(manager_yearly_parts, ignore_index=True).rename(columns={'group': 'AM'}).sort_values('AM')

dep_monthly = monthly_kpi_for_group(mart, report_months)
dep_yearly = yearly_weighted_kpi(dep_monthly, group_name='Отдел', year=2023)

print('Managers monthly rows:', len(manager_monthly))
print('Managers yearly rows:', len(manager_yearly))
manager_yearly.head()

Managers monthly rows: 120
Managers yearly rows: 10


,AM,year,numerator_1m,denominator_1m,coef_1m_year,numerator_2m,denominator_2m,coef_2m_year
0,Васильев Артем Александрович,2023,6119964.43,11832147.79,0.517232,482441.58,5818845.40,0.08291
1,Иванова Мария Сергеевна,2023,1660095.55,4727657.16,0.351146,0.00,2503864.75,0.00000
2,Кузнецов Михаил Иванович,2023,470182.98,982724.44,0.478448,0.00,167675.00,0.00000
3,Михайлов Андрей Сергеевич,2023,2235422.29,3361833.59,0.664941,0.00,1056004.68,0.00000
4,Петрова Анна Дмитриевна,2023,109442.52,98492.00,1.111182,0.00,0.00,NaN


## Шаг 4. Экспорт итоговых артефактов

Сохраняем:

- промежуточные таблицы для проверки,
- финальный отчет в Excel для руководителя.


In [4]:
from build_prolongation_report import add_excel_visuals

mart.to_csv(OUT_DIR / 'project_mart.csv', index=False)
manager_monthly.to_csv(OUT_DIR / 'manager_monthly_kpi.csv', index=False)
manager_yearly.to_csv(OUT_DIR / 'manager_yearly_kpi.csv', index=False)
dep_monthly.to_csv(OUT_DIR / 'department_monthly_kpi.csv', index=False)
dep_yearly.to_csv(OUT_DIR / 'department_yearly_kpi.csv', index=False)

report_path = OUT_DIR / 'prolongation_report_2023.xlsx'
with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
    manager_monthly.to_excel(writer, sheet_name='1_manager_kpi_monthly', index=False)
    manager_yearly.to_excel(writer, sheet_name='1_manager_kpi_yearly', index=False)
    dep_monthly.to_excel(writer, sheet_name='2_department_kpi', index=False)
    dep_yearly.to_excel(writer, sheet_name='2_department_yearly', index=False)

add_excel_visuals(report_path)

print('Saved report:', report_path)
print('Saved mart:', OUT_DIR / 'project_mart.csv')

Saved report: /Users/dmitrijpavlov/Desktop/DA/outputs/prolongation_report_2023.xlsx
Saved mart: /Users/dmitrijpavlov/Desktop/DA/outputs/project_mart.csv
